# Factor graphs — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## A factor graph makes every local scoring function visible as its own node
Factor graphs split a model into variables and factors. The examples multiply local factors, sum out variables, normalize beliefs, compare a factorized computation to a full table, and expose how a shared factor couples variables.

In [ ]:
# 1) multiply unary and pairwise factors for two binary variables
phi_x=np.array([0.7,0.3]); phi_y=np.array([0.4,0.6]); psi=np.array([[2.0,0.5],[0.5,1.5]])
score=phi_x[:,None]*psi*phi_y[None,:]; prob=score/score.sum()
plt.figure(figsize=(6,3)); plt.imshow(prob); plt.colorbar(); plt.title('normalized product of factors')
assert abs(prob.sum()-1)<1e-12 and prob[0,0]>prob[0,1]

In [ ]:
# 2) marginal belief for X by summing Y
bel_x=prob.sum(axis=1); plt.figure(figsize=(6,3)); plt.bar(['X=0','X=1'],bel_x); plt.title(f'belief X=0 = {bel_x[0]:.3f}')
assert abs(bel_x[0]-0.7)<1e-12

In [ ]:
# 3) factor-to-variable message sums over the other variable
msg_to_x=psi@phi_y; msg_to_x=msg_to_x/msg_to_x.sum()
plt.figure(figsize=(6,3)); plt.bar(['message to X=0','message to X=1'],msg_to_x); plt.title('factor message is a local sum')
assert abs(msg_to_x[0]-0.5)<1e-12

In [ ]:
# 4) full table equals factorized construction
full=np.zeros((2,2))
for x0,y0 in itertools.product([0,1],repeat=2): full[x0,y0]=phi_x[x0]*psi[x0,y0]*phi_y[y0]
plt.figure(figsize=(6,3)); plt.bar(['factorized','loop table'],[score.sum(),full.sum()]); plt.title('same unnormalized evidence')
assert np.allclose(full,score)

In [ ]:
# 5) changing one factor locally changes the belief
psi2=np.array([[4.0,0.2],[0.2,1.0]]); prob2=phi_x[:,None]*psi2*phi_y[None,:]; prob2=prob2/prob2.sum(); bel2=prob2.sum(1)
plt.figure(figsize=(6,3)); plt.bar(['old P(X=0)','new P(X=0)'],[bel_x[0],bel2[0]]); plt.title('local compatibility reshapes global belief')
assert bel2[0]>bel_x[0]